# MDA Text File Splitter

This notebook ingests MDA (Management Discussion & Analysis) text files from a folder and splits each file into separate segment files based on header taxonomy.

## 1. Configuration

Set your input and output folder paths here:

In [11]:
# === CONFIGURE YOUR PATHS HERE ===

INPUT_FOLDER = "Data"          # Folder containing your MDA text files
OUTPUT_FOLDER = "Data/mda_segments"      # Folder where segment files will be saved
FILE_PATTERN = "*.txt"                # Pattern to match MDA files (e.g., "*.txt", "*mda*.txt")

## 2. Import Libraries

In [12]:
import re
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
from glob import glob

## 3. Segment Taxonomy

This defines the patterns used to identify different MDA sections:

In [13]:
SEGMENT_TAXONOMY = [
    # === OVERVIEW ===
    ("overview", [
        r"executive\s+(?:overview|summary)",
        r"(?:business\s+)?overview",
        r"(?:key\s+)?highlights",
        r"introduction",
        r"summary\s+of\s+(?:results|operations|performance)",
        r"understanding\s+.*(?:financial\s+)?results",
        r"(?:our\s+)?business\s+(?:and\s+)?(?:overview|summary)",
    ]),
    # === RESULTS OF OPERATIONS ===
    ("results_of_operations", [
        r"results?\s+of\s+operations?",
        r"operating\s+results?",
        r"financial\s+results?",
        r"(?:discussion\s+of\s+)?(?:consolidated\s+)?results",
        r"performance\s+(?:summary|overview|review)",
    ]),
    # === REVENUE ===
    ("revenue", [
        r"(?:net\s+)?(?:revenues?|sales)(?:\s*$|\s+(?:and|&)\s+monetization)",
        r"(?:total\s+)?(?:revenues?|sales)\s+(?:discussion|analysis|overview)",
        r"revenue\s+(?:by\s+)?(?:segment|product|geography|region)",
        r"(?:net\s+)?sales\s*$",
        r"top.?line",
    ]),
    # === COSTS & EXPENSES ===
    ("costs_and_expenses", [
        r"costs?\s+(?:and|&)\s+expenses?",
        r"(?:total\s+)?(?:operating\s+)?expenses?(?:\s*$|\s+(?:discussion|analysis))",
        r"cost\s+of\s+(?:revenues?|sales|goods\s+sold)",
        r"cost\s+structure",
        r"expense\s+(?:discussion|analysis|overview)",
        r"(?:research\s+(?:and|&)\s+development|r\s*&\s*d)(?:\s+expenses?)?$",
        r"(?:selling,?\s+)?general\s+(?:and|&)\s+administrative",
        r"(?:sales\s+(?:and|&)\s+marketing|s\s*&\s*m)(?:\s+expenses?)?$",
        r"sg\s*&\s*a",
    ]),
    # === GROSS PROFIT / MARGIN ===
    ("gross_profit", [
        r"gross\s+(?:profit|margin|income)",
        r"cost\s+of\s+(?:revenues?|sales)",
    ]),
    # === OPERATING INCOME ===
    ("operating_income", [
        r"(?:income|loss)\s+from\s+operations?",
        r"operating\s+(?:income|loss|profit|margin)",
        r"operating\s+(?:results|performance)",
    ]),
    # === SEGMENT RESULTS ===
    ("segment_results", [
        r"segment\s+(?:results?|performance|profitability|information|analysis)",
        r"(?:business\s+)?segments?(?:\s+(?:results?|performance|review))?$",
        r"(?:reportable|operating)\s+segments?",
        r"results?\s+(?:of|by)\s+(?:reportable\s+)?segments?",
        r"(?:geographic|regional)\s+(?:results?|performance|breakdown)",
        r"(?:products?\s+(?:and|&)\s+services?|services?\s+(?:and|&)\s+products?)",
    ]),
    # === OTHER INCOME/EXPENSE ===
    ("other_income_expense", [
        r"other\s+(?:income|expense)(?:\s*,?\s*net)?",
        r"non.?operating\s+(?:income|expense|items?)",
        r"interest\s+(?:income|expense)(?:\s*,?\s*net)?",
        r"(?:investment|equity)\s+(?:income|gains?|losses?)",
    ]),
    # === INCOME TAXES ===
    ("income_taxes", [
        r"(?:provision|benefit)\s+for\s+(?:income\s+)?taxes",
        r"income\s+tax(?:es)?(?:\s+(?:expense|provision|benefit))?",
        r"(?:effective\s+)?tax\s+rate",
    ]),
    # === NET INCOME ===
    ("net_income", [
        r"net\s+(?:income|loss|earnings)",
        r"(?:earnings|income|loss)\s+per\s+share",
        r"(?:diluted|basic)\s+(?:eps|earnings)",
        r"bottom.?line",
    ]),
    # === FINANCIAL CONDITION / BALANCE SHEET ===
    ("financial_condition", [
        r"financial\s+(?:condition|position)",
        r"(?:consolidated\s+)?balance\s+sheet(?:\s+(?:discussion|analysis|review))?",
        r"(?:selected\s+)?financial\s+(?:data|information|position)",
        r"assets?\s+(?:and|&)\s+liabilities?",
    ]),
    # === LIQUIDITY ===
    ("liquidity", [
        r"liquidity(?:\s+(?:and|&)\s+capital\s+resources?)?",
        r"liquidity(?:\s+(?:and|&)\s+(?:financial|material)\s+(?:condition|resources?|requirements?))?",
        r"(?:sources?\s+(?:and|&)\s+uses?\s+of|uses?\s+(?:and|&)\s+sources?\s+of)\s+(?:cash|funds|liquidity)",
        r"(?:cash|liquidity)\s+(?:position|resources?|requirements?|availability)",
        r"(?:material\s+)?cash\s+requirements?",
        r"working\s+capital",
        r"capital\s+resources?(?:\s+(?:and|&)\s+liquidity)?",
    ]),
    # === CASH FLOW ===
    ("cash_flow", [
        r"cash\s+flows?(?:\s+(?:discussion|analysis|summary|overview))?",
        r"(?:consolidated\s+)?(?:statements?\s+of\s+)?cash\s+flows?",
        r"(?:cash\s+(?:provided|used|generated)\s+(?:by|in|from)\s+)?(?:operating|investing|financing)\s+activities",
        r"(?:operating|free)\s+cash\s+flow",
    ]),
    # === CAPITAL EXPENDITURES ===
    ("capital_expenditures", [
        r"capital\s+expenditures?(?:\s+(?:and|&)\s+leases?)?",
        r"cap\s*ex",
        r"(?:property|pp)\s*(?:and|&)\s*(?:equipment|e)",
        r"(?:capital|infrastructure)\s+(?:investments?|spending)",
        r"(?:additions?\s+to\s+)?(?:property|fixed\s+assets?)",
    ]),
    # === DEBT & FINANCING ===
    ("debt_and_financing", [
        r"(?:debt|financing)(?:\s+(?:and|&)\s+(?:capital|financing|liquidity))?",
        r"(?:long|short).?term\s+(?:debt|borrowings?)",
        r"(?:credit\s+)?(?:facilities?|agreements?|arrangements?)",
        r"(?:senior\s+)?(?:secured|unsecured)?\s*(?:notes?|bonds?)",
        r"(?:capital|financing)\s+(?:structure|arrangements?|activities)",
        r"indebtedness",
    ]),
    # === CAPITAL RETURN ===
    ("capital_return", [
        r"(?:share|stock)\s+repurchase(?:s|\s+program)?",
        r"(?:stock\s+)?buyback",
        r"dividends?(?:\s+(?:program|policy|payments?))?",
        r"(?:return\s+of\s+)?capital\s+(?:to\s+)?(?:shareholders?|stockholders?)",
        r"(?:capital|shareholder)\s+returns?",
    ]),
    # === COMMITMENTS & OBLIGATIONS ===
    ("commitments_and_obligations", [
        r"(?:commitments?|obligations?)(?:\s+(?:and|&)\s+(?:contingencies|obligations?|commitments?))?",
        r"(?:contractual|material|purchase)\s+(?:obligations?|commitments?)",
        r"off.?balance\s+sheet(?:\s+(?:arrangements?|items?|obligations?))?",
        r"(?:future|material)\s+(?:cash\s+)?(?:obligations?|requirements?|payments?)",
    ]),
    # === CONTINGENCIES & LEGAL ===
    ("contingencies", [
        r"(?:contingencies|contingent\s+liabilities?)",
        r"(?:legal|regulatory|litigation)\s+(?:matters?|proceedings?|contingencies?)",
        r"(?:claims?|lawsuits?|disputes?)",
        r"(?:regulatory|government)\s+(?:investigations?|inquiries?|matters?)",
    ]),
    # === ACQUISITIONS & DIVESTITURES ===
    ("acquisitions_and_divestitures", [
        r"(?:acquisitions?|business\s+combinations?)(?:\s+(?:and|&)\s+divestitures?)?",
        r"(?:mergers?\s+(?:and|&)\s+)?acquisitions?",
        r"(?:pending|completed|recent)\s+(?:acquisitions?|transactions?)",
        r"divestitures?|dispositions?",
        r"(?:strategic\s+)?(?:investments?|transactions?)",
    ]),
    # === CRITICAL ACCOUNTING ===
    ("critical_accounting", [
        r"critical\s+accounting\s+(?:policies|estimates?|judgments?)",
        r"significant\s+(?:accounting\s+)?(?:policies|estimates?|judgments?)",
        r"(?:key|significant)\s+(?:estimates?|assumptions?|judgments?)",
        r"(?:use\s+of\s+)?(?:management\s+)?estimates?",
        r"recently\s+(?:adopted|issued)\s+accounting\s+(?:standards?|pronouncements?|guidance)",
    ]),
    # === MARKET RISK ===
    ("market_risk", [
        r"(?:quantitative\s+(?:and|&)\s+qualitative\s+)?(?:disclosures?\s+about\s+)?market\s+risk",
        r"(?:market|financial)\s+risk(?:\s+(?:management|exposures?|factors?))?",
        r"(?:interest\s+rate|foreign\s+(?:currency|exchange)|commodity|equity)\s+risk",
        r"(?:derivative|hedging)\s+(?:instruments?|activities|programs?)",
        r"risk\s+management",
    ]),
    # === NON-GAAP ===
    ("non_gaap", [
        r"non.?gaap(?:\s+(?:financial\s+)?measures?)?",
        r"(?:adjusted|non.?gaap)\s+(?:ebitda|earnings?|income|results?)",
        r"(?:constant\s+currency|currency.?neutral)",
        r"(?:reconciliation\s+(?:of|to)\s+)?(?:gaap|non.?gaap)",
        r"(?:free\s+cash\s+flow|fcf)",
    ]),
    # === MACRO / OUTLOOK ===
    ("macro_and_outlook", [
        r"(?:macroeconomic|economic|market|industry)\s+(?:conditions?|environment|trends?|factors?)",
        r"(?:business\s+)?outlook",
        r"(?:forward\s+)?guidance",
        r"(?:trends?|factors?)\s+affecting\s+(?:our\s+)?(?:business|results|operations)",
        r"(?:known\s+)?trends?\s+(?:and|&)\s+uncertainties",
    ]),
    # === FORWARD-LOOKING ===
    ("forward_looking", [
        r"forward.?looking\s+statements?",
        r"(?:cautionary\s+)?(?:note|statement)\s+(?:regarding|about|concerning)",
        r"safe\s+harbor",
    ]),
    # === CONTROLS ===
    ("controls_and_procedures", [
        r"(?:disclosure\s+)?controls?\s+(?:and|&)\s+procedures?",
        r"internal\s+control(?:s)?\s+(?:over\s+)?financial\s+reporting",
        r"(?:changes?\s+in\s+)?internal\s+controls?",
    ]),
]

# Compile taxonomy patterns for efficiency
COMPILED_TAXONOMY = [
    (segment_name, [re.compile(pattern, re.IGNORECASE) for pattern in patterns])
    for segment_name, patterns in SEGMENT_TAXONOMY
]

print(f"Loaded {len(COMPILED_TAXONOMY)} segment categories")

Loaded 25 segment categories


## 4. Helper Classes and Functions

In [14]:
@dataclass
class SegmentMatch:
    """Represents a matched segment in the MDA document."""
    segment_name: str
    header_text: str
    start_line: int
    end_line: Optional[int]
    content: str


def is_header_line(line: str) -> bool:
    """
    Determine if a line looks like a section header.
    Headers are typically short, may be in caps, and don't end with periods.
    """
    stripped = line.strip()
    if not stripped:
        return False
    
    # Headers are usually relatively short
    if len(stripped) > 150:
        return False
    
    # Headers typically don't end with periods (unless abbreviation)
    if stripped.endswith('.') and not stripped.endswith('Inc.') and not stripped.endswith('Corp.'):
        if len(stripped) > 50:
            return False
    
    return True


def match_segment(line: str) -> Optional[str]:
    """
    Check if a line matches any segment in the taxonomy.
    Returns the segment name if matched, None otherwise.
    """
    stripped = line.strip()
    
    if not is_header_line(stripped):
        return None
    
    for segment_name, patterns in COMPILED_TAXONOMY:
        for pattern in patterns:
            if pattern.search(stripped):
                return segment_name
    
    return None

In [6]:
def extract_segments(content: str) -> List[SegmentMatch]:
    """
    Parse the MDA content and extract segments based on header taxonomy.
    """
    lines = content.split('\n')
    segments: List[SegmentMatch] = []
    
    current_segment: Optional[SegmentMatch] = None
    
    for i, line in enumerate(lines):
        segment_name = match_segment(line)
        
        if segment_name:
            # Close the previous segment
            if current_segment:
                current_segment.end_line = i - 1
                segment_lines = lines[current_segment.start_line:current_segment.end_line + 1]
                current_segment.content = '\n'.join(segment_lines)
                segments.append(current_segment)
            
            # Start a new segment
            current_segment = SegmentMatch(
                segment_name=segment_name,
                header_text=line.strip(),
                start_line=i,
                end_line=None,
                content=""
            )
    
    # Close the last segment
    if current_segment:
        current_segment.end_line = len(lines) - 1
        segment_lines = lines[current_segment.start_line:current_segment.end_line + 1]
        current_segment.content = '\n'.join(segment_lines)
        segments.append(current_segment)
    
    return segments


def merge_consecutive_segments(segments: List[SegmentMatch]) -> List[SegmentMatch]:
    """
    Merge consecutive segments with the same segment name.
    """
    if not segments:
        return segments
    
    merged: List[SegmentMatch] = []
    current = segments[0]
    
    for segment in segments[1:]:
        if segment.segment_name == current.segment_name:
            # Merge: extend current segment
            current.end_line = segment.end_line
            current.content = current.content + '\n' + segment.content
        else:
            merged.append(current)
            current = segment
    
    merged.append(current)
    return merged

In [15]:
def save_segments(
    segments: List[SegmentMatch],
    output_dir: str,
    base_filename: str
) -> Dict[str, str]:
    """
    Save each segment to a separate text file.
    Returns a dictionary mapping segment names to their output file paths.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    output_files: Dict[str, str] = {}
    segment_counts: Dict[str, int] = {}
    
    for segment in segments:
        # Handle multiple segments with the same name
        count = segment_counts.get(segment.segment_name, 0)
        segment_counts[segment.segment_name] = count + 1
        
        # Create filename
        if count == 0:
            filename = f"{base_filename}_{segment.segment_name}.txt"
        else:
            filename = f"{base_filename}_{segment.segment_name}_{count + 1}.txt"
        
        filepath = os.path.join(output_dir, filename)
        
        # Write segment content
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"# Segment: {segment.segment_name}\n")
            f.write(f"# Original Header: {segment.header_text}\n")
            f.write(f"# Lines: {segment.start_line + 1} - {segment.end_line + 1}\n")
            f.write("#" + "=" * 60 + "\n\n")
            f.write(segment.content)
        
        output_files[segment.segment_name] = filepath
    
    return output_files

## 5. Main Processing Function

In [16]:
def process_mda_file(
    input_file: str,
    output_dir: str,
    merge_same_segments: bool = True
) -> Dict[str, any]:
    """
    Process a single MDA file and split it into segments.
    
    Returns a dictionary with processing results.
    """
    input_path = Path(input_file)
    base_filename = input_path.stem
    
    # Create subfolder for this file's segments
    file_output_dir = os.path.join(output_dir, base_filename)
    
    result = {
        "input_file": input_file,
        "output_dir": file_output_dir,
        "segments_found": 0,
        "segments_saved": 0,
        "segment_names": [],
        "output_files": {},
        "error": None
    }
    
    try:
        # Read input file
        with open(input_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Extract segments
        segments = extract_segments(content)
        result["segments_found"] = len(segments)
        
        if merge_same_segments:
            segments = merge_consecutive_segments(segments)
        
        if segments:
            # Save segments to files
            output_files = save_segments(segments, file_output_dir, base_filename)
            result["segments_saved"] = len(output_files)
            result["segment_names"] = [seg.segment_name for seg in segments]
            result["output_files"] = output_files
        
    except Exception as e:
        result["error"] = str(e)
    
    return result

## 6. Run the Splitter

This cell loops through all MDA files in the input folder and processes each one:

In [17]:
# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Find all MDA files in the input folder
input_path = Path(INPUT_FOLDER)
mda_files = list(input_path.glob(FILE_PATTERN))

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"File pattern: {FILE_PATTERN}")
print(f"Found {len(mda_files)} MDA files to process")
print("=" * 60)

Input folder: Data
Output folder: Data/mda_segments
File pattern: *.txt
Found 5 MDA files to process


In [10]:
# Process all files
all_results = []

for i, mda_file in enumerate(mda_files, 1):
    print(f"\n[{i}/{len(mda_files)}] Processing: {mda_file.name}")
    print("-" * 40)
    
    result = process_mda_file(str(mda_file), OUTPUT_FOLDER)
    all_results.append(result)
    
    if result["error"]:
        print(f"  ERROR: {result['error']}")
    else:
        print(f"  Segments found: {result['segments_found']}")
        print(f"  Segments saved: {result['segments_saved']}")
        if result["segment_names"]:
            print(f"  Segment types: {', '.join(set(result['segment_names']))}")
        print(f"  Output: {result['output_dir']}")

print("\n" + "=" * 60)
print("PROCESSING COMPLETE")


[1/5] Processing: Alphabet Inc._10-Q_2025-09-30T00_00_00_English_MDA.txt
----------------------------------------
  Segments found: 0
  Segments saved: 0
  Output: Dara/mda_segments/Alphabet Inc._10-Q_2025-09-30T00_00_00_English_MDA

[2/5] Processing: Apple Inc._10-Q_2025-12-27T00_00_00_English_MDA.txt
----------------------------------------
  Segments found: 0
  Segments saved: 0
  Output: Dara/mda_segments/Apple Inc._10-Q_2025-12-27T00_00_00_English_MDA

[3/5] Processing: Microsoft Corporation_10-Q_2025-12-31T00_00_00_English_MDA.txt
----------------------------------------
  Segments found: 0
  Segments saved: 0
  Output: Dara/mda_segments/Microsoft Corporation_10-Q_2025-12-31T00_00_00_English_MDA

[4/5] Processing: NVIDIA Corporation_10-Q_2025-10-26T00_00_00_English_MDA.txt
----------------------------------------
  Segments found: 0
  Segments saved: 0
  Output: Dara/mda_segments/NVIDIA Corporation_10-Q_2025-10-26T00_00_00_English_MDA

[5/5] Processing: Amazon.com, Inc._10-Q_202

## 7. Summary Report

In [ ]:
# Generate summary
total_files = len(all_results)
successful = sum(1 for r in all_results if r["error"] is None)
failed = total_files - successful
total_segments = sum(r["segments_saved"] for r in all_results)

print("\n" + "=" * 60)
print("SUMMARY REPORT")
print("=" * 60)
print(f"Total files processed: {total_files}")
print(f"Successful: {successful}")
print(f"Failed: {failed}")
print(f"Total segments created: {total_segments}")
print(f"\nOutput location: {OUTPUT_FOLDER}")

# Count segment types across all files
all_segment_types = {}
for r in all_results:
    for seg_name in r["segment_names"]:
        all_segment_types[seg_name] = all_segment_types.get(seg_name, 0) + 1

if all_segment_types:
    print("\nSegment type distribution:")
    for seg_type, count in sorted(all_segment_types.items(), key=lambda x: -x[1]):
        print(f"  {seg_type}: {count}")

In [ ]:
# Display detailed results as a table
print("\nDetailed Results:")
print("-" * 80)
print(f"{'File':<40} {'Segments':<10} {'Status':<10}")
print("-" * 80)

for r in all_results:
    filename = Path(r["input_file"]).name[:38]
    segments = r["segments_saved"]
    status = "ERROR" if r["error"] else "OK"
    print(f"{filename:<40} {segments:<10} {status:<10}")

## 8. View Sample Output (Optional)

Run this cell to preview the first segment from the first file:

In [ ]:
# Preview first segment from first successful file
for r in all_results:
    if r["output_files"]:
        first_segment_file = list(r["output_files"].values())[0]
        print(f"Preview of: {first_segment_file}")
        print("=" * 60)
        with open(first_segment_file, 'r', encoding='utf-8') as f:
            content = f.read()
            # Show first 1000 characters
            print(content[:1000])
            if len(content) > 1000:
                print(f"\n... [truncated, total {len(content)} characters]")
        break
else:
    print("No segments were created.")